# 14 — Multilingual Keyword Extraction with KeyBERT

## HZ Knowledge Management Platform (KMP)

### Objective

Develop and evaluate a multilingual keyword extraction method for Dutch and English research publications using KeyBERT.

### Business rules

1. If author-provided keywords exist, preserve them and suggest five additional, non-duplicate keywords.
2. If no author-provided keywords exist, suggest ten keywords.
3. Keep extracted author keywords and generated suggestions in separate output fields.
4. Flag uncertain cases for human review.

### Methodology

- Reuse the existing preprocessing pipeline.
- Use the shared `sample_manifest.json` development/test split.
- Use a multilingual sentence-transformer model with KeyBERT.
- Compare against the existing YAKE baseline.
- Evaluate relevance, language correctness, redundancy, and semantic similarity.

### Reproducibility and privacy

The model will run locally after its licence and download requirements are reviewed. Research PDFs and extracted text must not be sent to unapproved external services. The test split will be reserved for final evaluation.

**Note:** KeyBERT extracts phrases from the document text. It does not generate entirely new concepts that are absent from the source.

In [1]:
# 1. Import required libraries

import json
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("All imports successful!")

/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 17:27:52.630000 20020 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Python version: 3.11.18
Pandas version: 3.1.0
PyTorch version: 2.15.0
MPS available: True
All imports successful!


## Load the shared preprocessing data

This experiment reuses the document inventory, extracted PDF sections, and section-level language detection produced by the existing preprocessing pipeline.

The files are joined using `document_id` and `section_number`. This ensures that KeyBERT uses the same source documents as the other keyword extraction experiments.

In [2]:
from pathlib import Path

# Find the main DDD project folder
current_dir = Path.cwd()

if (current_dir / "sample_manifest.json").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "sample_manifest.json").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate the DDD project folder.")

inventory_path = PROJECT_ROOT / "output" / "inventory" / "documents.csv"
sections_path = PROJECT_ROOT / "output" / "preprocessing" / "sections.jsonl"
languages_path = PROJECT_ROOT / "output" / "preprocessing" / "section_languages.csv"

for path in [inventory_path, sections_path, languages_path]:
    print(path.name, "— Found:", path.is_file())

documents.csv — Found: True
sections.jsonl — Found: True
section_languages.csv — Found: True


In [3]:
import pandas as pd

documents_df = pd.read_csv(inventory_path)
sections_df = pd.read_json(sections_path, lines=True)
languages_df = pd.read_csv(languages_path)

print("Documents:", len(documents_df))
print("Sections:", len(sections_df))
print("Section language records:", len(languages_df))

display(sections_df.head())

Documents: 93
Sections: 3327
Section language records: 3327


,document_id,file_name,section_number,section_label,page_start,page_end,original_text,character_count
0,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,1,front_matter,1,1,water\n\n\n\nArticle\nAdaptive Capacities f...,2148
1,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,2,introduction,1,1,1. Introduction\n\n Climate change will res...,1680
2,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,3,introduction,2,2,"Water 2019, 11, 2643 ...",3737
3,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,4,introduction,3,3,"Water 2019, 11, 2643 ...",5150
4,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,5,introduction,4,4,"Water 2019, 11, 2643 ...",2329


In [4]:
join_columns = ["document_id", "section_number"]

assert not sections_df.duplicated(join_columns).any()
assert not languages_df.duplicated(join_columns).any()

sections_with_language_df = sections_df.merge(
    languages_df[
        ["document_id", "section_number",
         "detected_language", "language_score"]
    ],
    on=join_columns,
    how="left",
    validate="one_to_one"
)

print("Combined sections:", len(sections_with_language_df))
print(
    "Missing language labels:",
    sections_with_language_df["detected_language"].isna().sum()
)

display(sections_with_language_df.head())

Combined sections: 3327
Missing language labels: 0


,document_id,file_name,section_number,section_label,page_start,page_end,original_text,character_count,detected_language,language_score
0,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,1,front_matter,1,1,water\n\n\n\nArticle\nAdaptive Capacities f...,2148,en,0.879993
1,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,2,introduction,1,1,1. Introduction\n\n Climate change will res...,1680,en,0.912203
2,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,3,introduction,2,2,"Water 2019, 11, 2643 ...",3737,en,0.924616
3,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,4,introduction,3,3,"Water 2019, 11, 2643 ...",5150,en,0.883415
4,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,5,introduction,4,4,"Water 2019, 11, 2643 ...",2329,en,0.926408


## 6. Inspect the shared experiment sample

The full research collection contains 93 documents.

For the initial comparison between keyword extraction methods, the project uses a smaller sample defined in `sample_manifest.json`.

This section checks which documents belong to the development and test groups. The development group is used for model configuration, while the test group is reserved for final evaluation.

In [5]:
# Load and inspect the shared experiment sample

import json

manifest_path = PROJECT_ROOT / "sample_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as file:
    manifest = json.load(file)

sample_df = pd.DataFrame(manifest["rows"])

print("Total documents in full collection:", len(documents_df))
print("Total documents in experiment sample:", len(sample_df))

print("\nManifest columns:")
print(sample_df.columns.tolist())

display(sample_df)

Total documents in full collection: 93
Total documents in experiment sample: 12

Manifest columns:
['document_id', 'file_name', 'language', 'split']


,document_id,file_name,language,split
0,doc_5a6ef3b4929cdb64,005_building-relationships-in-higher-education...,en,dev
1,doc_a31197553fa562f4,072_overflow-tests-dutch-levees-annexes.pdf,en,dev
2,doc_1284c81a29b16bf1,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en,dev
3,doc_3b49863c125ac95b,017_integrating-text-structure-instruction-in-...,en,test
4,doc_02ab54e64376516f,071_overflow-tests-belgian-levees-annexes.pdf,en,test
5,doc_f6e98034f8bc51ea,020_long-term-forecast-of-thermal-mortality-wi...,en,test
6,doc_536f964e25d848c9,062_leven-in-zeeland-2021.pdf,nl,dev
7,doc_c136a22677a363ed,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl,dev
8,doc_2aff122389509a84,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl,dev
9,doc_6a489106f53933b1,056_jonge-zeeuwen-in-beweging.pdf,nl,test


In [6]:
# Check the development/test distribution

print("Documents per split:")
display(sample_df["split"].value_counts())

print("\nDocuments per language and split:")
display(
    pd.crosstab(
        sample_df["language"],
        sample_df["split"]
    )
)

Documents per split:


split
dev     6
test    6
Name: count, dtype: int64


Documents per language and split:


split,dev,test
language,,
en,3,3
nl,3,3


## 7. Select development documents

The shared manifest contains 12 research documents: six for development and six for final testing.

During development, only the six documents marked `dev` will be used. The test documents will remain separate to avoid influencing model configuration.

The development set contains three English and three Dutch publications.

In [7]:
# Select development and test document IDs

dev_df = sample_df[
    sample_df["split"] == "dev"
].copy()

test_df = sample_df[
    sample_df["split"] == "test"
].copy()

# Check that no document appears in both groups
assert set(dev_df["document_id"]).isdisjoint(
    set(test_df["document_id"])
)

print("Development documents:", len(dev_df))
print("Reserved test documents:", len(test_df))

display(
    dev_df[["document_id", "file_name", "language"]]
)

Development documents: 6
Reserved test documents: 6


,document_id,file_name,language
0,doc_5a6ef3b4929cdb64,005_building-relationships-in-higher-education...,en
1,doc_a31197553fa562f4,072_overflow-tests-dutch-levees-annexes.pdf,en
2,doc_1284c81a29b16bf1,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en
6,doc_536f964e25d848c9,062_leven-in-zeeland-2021.pdf,nl
7,doc_c136a22677a363ed,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl
8,doc_2aff122389509a84,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl


## 8. Identify relevant document sections

The preprocessing pipeline divides research publications into sections such as abstracts, introductions, results, and conclusions.

We first identify all sections belonging to the development documents. This allows us to prepare suitable input text for KeyBERT without processing the reserved test documents.

In [8]:
# Select sections from development documents only

dev_sections_df = sections_with_language_df[
    sections_with_language_df["document_id"].isin(
        dev_df["document_id"]
    )
].copy()

print("Total development sections:", len(dev_sections_df))

print("\nSection labels available:")
display(
    dev_sections_df["section_label"].value_counts()
)

print("\nSections per development document:")
display(
    dev_sections_df.groupby("document_id")
    .size()
    .reset_index(name="section_count")
)

Total development sections: 344

Section labels available:


section_label
unknown         130
introduction    103
methods          36
conclusion       23
discussion       17
front_matter     13
keywords          7
results           7
references        4
contents          4
Name: count, dtype: int64


Sections per development document:


,document_id,section_count
0,doc_1284c81a29b16bf1,10
1,doc_2aff122389509a84,47
2,doc_536f964e25d848c9,126
3,doc_5a6ef3b4929cdb64,30
4,doc_a31197553fa562f4,111
5,doc_c136a22677a363ed,20


In [9]:
# Inspect the actual section labels in the development set

section_labels = sorted(
    dev_sections_df["section_label"]
    .dropna()
    .unique()
    .tolist()
)

print("Available section labels:")
for label in section_labels:
    print("-", label)

print("\nExample section text:")
if not dev_sections_df.empty:
    example = dev_sections_df.iloc[0]
    print("Document:", example["file_name"])
    print("Section:", example["section_label"])
    print(str(example["original_text"])[:800])

Available section labels:
- conclusion
- contents
- discussion
- front_matter
- introduction
- keywords
- methods
- references
- results
- unknown

Example section text:
Document: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
Section: front_matter
             Utrecht University Repository



Title                       Building relationships in higher education to support students’ motivation


Authors                  Leenknecht, Martijn J.M.; Snijders, Ingrid; Wijnia, Lisette; Rikers, Remy M.J.P.;
                          Loyens, Sofie M.M.


Published in              Teaching in Higher Education


Publication Date          2023


Link                       http://hdl.handle.net/1874/431290


Citation                  Leenknecht, M J M, Snijders, I, Wijnia, L, Rikers, R M J P & Loyens, S M
               M 2023, 'Building relationships in higher education to support students’
                             motivation', Teaching in Higher Education,

## 9. Prepare source text for KeyBERT

The six development documents are prepared using the shared preprocessing data.

The initial input selection follows the existing keyword experiment:

- Prioritize conclusion, introduction, discussion, results, methods, front matter, and unknown sections.
- Include abstract sections if available.
- Exclude references and tables of contents.
- Keep author-provided keyword sections separate from the KeyBERT input.
- Limit individual section excerpts to 1,500 characters.
- Limit the combined document input to 6,000 characters.

The same source preparation should be used for YAKE and KeyBERT to ensure a fair comparison.

In [10]:
# Shared source-text preparation settings

MAX_SOURCE_CHARS = 6000
MAX_SECTION_CHARS = 1500

SECTION_PRIORITY = [
    "abstract",
    "conclusion",
    "introduction",
    "discussion",
    "results",
    "methods",
    "front_matter",
    "unknown"
]

# Exclude author keywords, references, and contents
excluded_labels = [
    "keywords",
    "references",
    "contents"
]

priority_map = {
    label: index
    for index, label in enumerate(SECTION_PRIORITY)
}

# Select useful sections from development documents
selected_sections_df = dev_sections_df[
    dev_sections_df["section_label"].isin(SECTION_PRIORITY)
].copy()

selected_sections_df["priority"] = (
    selected_sections_df["section_label"].map(priority_map)
)

selected_sections_df = selected_sections_df.sort_values(
    ["document_id", "priority", "section_number"]
)

print("Original development sections:", len(dev_sections_df))
print("Selected sections:", len(selected_sections_df))
print("Excluded sections:", len(dev_sections_df) - len(selected_sections_df))

display(selected_sections_df["section_label"].value_counts())

Original development sections: 344
Selected sections: 329
Excluded sections: 15


section_label
unknown         130
introduction    103
methods          36
conclusion       23
discussion       17
front_matter     13
results           7
Name: count, dtype: int64

In [11]:
def prepare_keybert_text(document_id):
    """Prepare a maximum of 6000 characters for one document."""

    document_sections = selected_sections_df[
        selected_sections_df["document_id"] == document_id
    ]

    text_parts = []
    remaining_chars = MAX_SOURCE_CHARS

    for _, section in document_sections.iterrows():

        text = section["original_text"]

        if not isinstance(text, str):
            continue

        # Normalize whitespace
        cleaned_text = " ".join(text.split())

        if not cleaned_text:
            continue

        # Limit each section to 1500 characters
        section_text = cleaned_text[:MAX_SECTION_CHARS]

        # Reserve two characters for the separator
        separator = "\n\n" if text_parts else ""
        available = remaining_chars - len(separator)

        if available <= 0:
            break

        section_text = section_text[:available]

        text_parts.append(separator + section_text)
        remaining_chars -= len(separator) + len(section_text)

        if remaining_chars <= 0:
            break

    return "".join(text_parts)


# Build the input table for the six development documents
keybert_dev_df = dev_df[
    ["document_id", "file_name", "language"]
].copy()

keybert_dev_df["source_text"] = (
    keybert_dev_df["document_id"].apply(prepare_keybert_text)
)

keybert_dev_df["source_characters"] = (
    keybert_dev_df["source_text"].str.len()
)

display(
    keybert_dev_df[
        ["file_name", "language", "source_characters"]
    ]
)

,file_name,language,source_characters
0,005_building-relationships-in-higher-education...,en,6000
1,072_overflow-tests-dutch-levees-annexes.pdf,en,6000
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en,6000
6,062_leven-in-zeeland-2021.pdf,nl,6000
7,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl,6000
8,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl,6000


In [12]:
# Check that the prepared inputs are valid

assert len(keybert_dev_df) == 6

assert (
    keybert_dev_df["source_characters"] <= MAX_SOURCE_CHARS
).all()

assert (
    keybert_dev_df["source_characters"] > 0
).all()

assert set(keybert_dev_df["document_id"]).isdisjoint(
    set(test_df["document_id"])
)

print("All six development documents have valid input text.")
print("Maximum input length:", keybert_dev_df["source_characters"].max())
print("Test documents remain excluded.")

print("\nInput lengths by document:")
display(
    keybert_dev_df[
        ["file_name", "language", "source_characters"]
    ]
)

All six development documents have valid input text.
Maximum input length: 6000
Test documents remain excluded.

Input lengths by document:


,file_name,language,source_characters
0,005_building-relationships-in-higher-education...,en,6000
1,072_overflow-tests-dutch-levees-annexes.pdf,en,6000
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en,6000
6,062_leven-in-zeeland-2021.pdf,nl,6000
7,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl,6000
8,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl,6000


In [13]:
example = keybert_dev_df.iloc[0]

print("Document:", example["file_name"])
print("Language:", example["language"])
print("Input length:", example["source_characters"])

print("\n--- FIRST 1500 CHARACTERS ---\n")
print(example["source_text"][:1500])

Document: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
Language: en
Input length: 6000

--- FIRST 1500 CHARACTERS ---

Conclusion Students’ motivation is ignited by their social context (e.g. Connell and Wellborn 1991; Ryan and Deci 2017; Skinner and Belmont 1993). In HE, two embedded social contexts can be distinguished: in-class teaching approaches and student-staﬀrelationships in the HE-institute. Students’ perceptions of both social contexts are associated with their motivation. Especially the aﬀective component of the student-staﬀrelationship quality was found to be important for students’ motivation, as perceived aﬀective commitment and involvement predicted students’ more autonomous forms of motivation (i.e. intrin- sic and identiﬁed motivation). Structure was found to be negatively associated with con- trolled forms of motivation. The embedded contexts were found to interact in their association with students’ motivation. The association bet

## 10. Initialize multilingual KeyBERT

KeyBERT identifies candidate keywords and keyphrases from document text using semantic similarity.

This experiment uses the multilingual sentence-transformer model `paraphrase-multilingual-MiniLM-L12-v2`, which supports both Dutch and English.

The initial configuration extracts phrases containing one to three words and uses Maximal Marginal Relevance (MMR) to encourage diversity.

This is an initial development experiment. Model settings will be evaluated before being finalized.

In [14]:
# Initialize the multilingual embedding model

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

# Connect the embedding model to KeyBERT
kw_model = KeyBERT(model=embedding_model)

print("Multilingual embedding model loaded successfully!")
print("Model:", MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24209.37it/s]


Multilingual embedding model loaded successfully!
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 11. First KeyBERT extraction experiment

The first development document is used to verify that multilingual KeyBERT can extract meaningful keywords from the prepared source text.

The model initially returns 30 candidate keyphrases. These are exploratory candidates, not the final five or ten suggestions required by the business rules.

In [15]:
# Select the first English development document

example_document = keybert_dev_df.iloc[0]

document_text = example_document["source_text"]

print("Document:", example_document["file_name"])
print("Language:", example_document["language"])
print("Input characters:", len(document_text))

# Extract 30 candidate keyphrases
candidate_keywords = kw_model.extract_keywords(
    document_text,
    keyphrase_ngram_range=(1, 3),
    stop_words=None,
    use_mmr=True,
    diversity=0.3,
    top_n=30
)

print("\nNumber of candidate keywords:", len(candidate_keywords))

display(
    pd.DataFrame(
        candidate_keywords,
        columns=["keyword", "similarity_score"]
    )
)

Document: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
Language: en
Input characters: 6000

Number of candidate keywords: 30


,keyword,similarity_score
0,conclusion students motivation,0.7318
1,aﬀect students motivation,0.6941
2,students motivation was,0.6906
3,and students motivation,0.6827
4,students motivation of,0.6816
5,students motivation is,0.6741
6,motivation and students,0.6738
7,between students perceptions,0.6720
8,both students perceptions,0.6680
9,students motivation but,0.6675


In [16]:
# Validate the initial extraction

print("Candidate count:", len(candidate_keywords))

print("\nTop 10 candidates:")
for rank, (keyword, score) in enumerate(
    candidate_keywords[:10], start=1
):
    print(f"{rank}. {keyword} — {score:.4f}")

Candidate count: 30

Top 10 candidates:
1. conclusion students motivation — 0.7318
2. aﬀect students motivation — 0.6941
3. students motivation was — 0.6906
4. and students motivation — 0.6827
5. students motivation of — 0.6816
6. students motivation is — 0.6741
7. motivation and students — 0.6738
8. between students perceptions — 0.6720
9. both students perceptions — 0.6680
10. students motivation but — 0.6675


## 12. Improve candidate keyword quality

### Initial observation

The first KeyBERT experiment successfully extracted 30 candidate keyphrases from an English development document.

However, many candidates contained stop words, incomplete phrases, repeated concepts, section headings, and PDF extraction artifacts.

Examples include:

- `conclusion students motivation`
- `students motivation was`
- `and students motivation`
- `students motiv ation`

The initial results suggest that candidate generation and text cleaning need improvement before the method can produce useful keyword suggestions.

### Improvement strategy

The next experiment will:

1. Remove common stop words from English candidate phrases.
2. Retain one-to-three-word candidate phrases.
3. Continue using multilingual sentence embeddings.
4. Compare the improved candidates with the initial results.

Dutch stop-word handling will be tested separately.

In [17]:
# Improved English candidate extraction

improved_keywords_en = kw_model.extract_keywords(
    document_text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    use_mmr=True,
    diversity=0.3,
    top_n=30
)

improved_keywords_df = pd.DataFrame(
    improved_keywords_en,
    columns=["keyword", "similarity_score"]
)

print("Improved English candidate count:", len(improved_keywords_en))

display(improved_keywords_df)

Improved English candidate count: 30


,keyword,similarity_score
0,conclusion students motivation,0.7318
1,students motivation perceived,0.7067
2,motivation students perceptions,0.7066
3,students motivation context,0.7046
4,students motivation associated,0.7017
5,students motivation interplay,0.6942
6,aﬀect students motivation,0.6941
7,associated students motivation,0.6891
8,important students motivation,0.6883
9,students motivation mediated,0.6801


In [18]:
# Compare initial and improved candidates

comparison_df = pd.DataFrame({
    "Initial candidates": [
        keyword for keyword, score in candidate_keywords[:10]
    ],
    "Improved candidates": [
        keyword for keyword, score in improved_keywords_en[:10]
    ]
})

display(comparison_df)

,Initial candidates,Improved candidates
0,conclusion students motivation,conclusion students motivation
1,aﬀect students motivation,students motivation perceived
2,students motivation was,motivation students perceptions
3,and students motivation,students motivation context
4,students motivation of,students motivation associated
5,students motivation is,students motivation interplay
6,motivation and students,aﬀect students motivation
7,between students perceptions,associated students motivation
8,both students perceptions,important students motivation
9,students motivation but,students motivation mediated


In [19]:
# Inspect the top 10 improved candidates

print("Top 10 improved English keywords:\n")

for rank, (keyword, score) in enumerate(
    improved_keywords_en[:10],
    start=1
):
    print(f"{rank}. {keyword} — {score:.4f}")

Top 10 improved English keywords:

1. conclusion students motivation — 0.7318
2. students motivation perceived — 0.7067
3. motivation students perceptions — 0.7066
4. students motivation context — 0.7046
5. students motivation associated — 0.7017
6. students motivation interplay — 0.6942
7. aﬀect students motivation — 0.6941
8. associated students motivation — 0.6891
9. important students motivation — 0.6883
10. students motivation mediated — 0.6801


## 13. Experiment C: Improve phrase quality and diversity

The English stop-word experiment reduced incomplete phrases containing common words, but the results remained repetitive.

This experiment tests two changes:

1. Restrict candidate phrases to one or two words.
2. Increase MMR diversity from 0.3 to 0.6.

The aim is to identify a broader range of meaningful research concepts rather than repeatedly selecting variations of the same phrase.

These settings are experimental and will be evaluated on the development documents before final selection.

In [20]:
# Experiment C: shorter phrases and stronger diversity

experiment_c_keywords = kw_model.extract_keywords(
    document_text,
    keyphrase_ngram_range=(1, 2),
    stop_words="english",
    use_mmr=True,
    diversity=0.6,
    top_n=30
)

experiment_c_df = pd.DataFrame(
    experiment_c_keywords,
    columns=["keyword", "similarity_score"]
)

display(experiment_c_df)

,keyword,similarity_score
0,students motivation,0.6466
1,perceptions aﬀective,0.4977
2,teacher relationship,0.4888
3,socializing aspect,0.4595
4,distinguished class,0.4088
5,involvement logical,0.3860
6,results associations,0.3828
7,conclusion,0.3821
8,context insti,0.3663
9,emotional bonds,0.3532


In [21]:
# Compare the three experiments

comparison_abc = pd.DataFrame({
    "A: Initial": [
        keyword for keyword, score in candidate_keywords[:10]
    ],
    "B: Stop words": [
        keyword for keyword, score in improved_keywords_en[:10]
    ],
    "C: Shorter + diverse": [
        keyword for keyword, score in experiment_c_keywords[:10]
    ]
})

display(comparison_abc)

,A: Initial,B: Stop words,C: Shorter + diverse
0,conclusion students motivation,conclusion students motivation,students motivation
1,aﬀect students motivation,students motivation perceived,perceptions aﬀective
2,students motivation was,motivation students perceptions,teacher relationship
3,and students motivation,students motivation context,socializing aspect
4,students motivation of,students motivation associated,distinguished class
5,students motivation is,students motivation interplay,involvement logical
6,motivation and students,aﬀect students motivation,results associations
7,between students perceptions,associated students motivation,conclusion
8,both students perceptions,important students motivation,context insti
9,students motivation but,students motivation mediated,emotional bonds


In [22]:
print("Experiment C — Top 10 candidates:\n")

for rank, (keyword, score) in enumerate(
    experiment_c_keywords[:10], start=1
):
    print(f"{rank}. {keyword} — {score:.4f}")

Experiment C — Top 10 candidates:

1. students motivation — 0.6466
2. perceptions aﬀective — 0.4977
3. teacher relationship — 0.4888
4. socializing aspect — 0.4595
5. distinguished class — 0.4088
6. involvement logical — 0.3860
7. results associations — 0.3828
8. conclusion — 0.3821
9. context insti — 0.3663
10. emotional bonds — 0.3532


## 14. Inspect author-provided keywords

The project requires author-provided keywords to be preserved separately from new KeyBERT suggestions.

The preprocessing data contains sections labeled `keywords`. These sections must be inspected before deciding whether a document requires five additional suggestions or ten new suggestions.

This step examines the original keyword-section text without modifying it.

In [24]:
# Find keyword sections in the development documents

author_keyword_sections_df = dev_sections_df[
    dev_sections_df["section_label"] == "keywords"
].copy()

print(
    "Keyword sections found:",
    len(author_keyword_sections_df)
)

print(
    "Development documents with keyword sections:",
    author_keyword_sections_df["document_id"].nunique()
)

# Display the original keyword-section text
for _, row in author_keyword_sections_df.iterrows():

    print("\n" + "=" * 70)
    print("Document:", row["file_name"])
    print("Section number:", row["section_number"])
    print("Original text:")
    print(str(row["original_text"])[:1500])

Keyword sections found: 7
Development documents with keyword sections: 1

Document: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
Section number: 4
Original text:
                                                        KEYWORDS
   situated. The  interplay between the two  contexts and  their
                                                                                        Student-staﬀinteractions;
   association with students’ motivation was studied in the current                                                                                   need-supportive teaching;
   study. In a broad sample (N=597), perceptions of student-staﬀ      relationship quality;
   relationship  quality,  need-supportive  teaching  in  class,  and      motivation; self-
  autonomous  and   controlled   motivation  were  measured.     determination theory
   Perceptions of in-class teacher involvement were associated with
  autonomous motivation, while perceptions

In [25]:
import unicodedata

examples = [
    "aﬀect students motivation",
    "signiﬁcant predictors",
    "staﬀ relationship quality"
]

for phrase in examples:
    cleaned = unicodedata.normalize("NFKC", phrase)

    print("Before:", phrase)
    print("After: ", cleaned)
    print()

Before: aﬀect students motivation
After:  affect students motivation

Before: signiﬁcant predictors
After:  significant predictors

Before: staﬀ relationship quality
After:  staff relationship quality



## 15. Check author-keyword coverage

The section-label inspection showed that seven sections were labeled `keywords`, but all belonged to one document.

Some sections may have been mislabeled during PDF segmentation. Therefore, the absence of a `keywords` section does not prove that author keywords are absent.

This step searches the development documents for common English and Dutch keyword headings before applying the five-versus-ten suggestion rule.

In [26]:
import re

keyword_heading_pattern = re.compile(
    r"\b(keywords|key words|index terms|"
    r"trefwoorden|sleutelwoorden|kernwoorden)\b",
    flags=re.IGNORECASE
)

keyword_matches = []

for _, row in dev_sections_df.iterrows():
    text = row["original_text"]

    if not isinstance(text, str):
        continue

    match = keyword_heading_pattern.search(text)

    if match:
        start = max(0, match.start() - 80)
        end = min(len(text), match.end() + 350)

        keyword_matches.append({
            "document_id": row["document_id"],
            "file_name": row["file_name"],
            "section_number": row["section_number"],
            "section_label": row["section_label"],
            "matched_heading": match.group(),
            "text_excerpt": text[start:end]
        })

keyword_matches_df = pd.DataFrame(keyword_matches)

print("Potential keyword-heading matches:", len(keyword_matches_df))

if not keyword_matches_df.empty:
    display(
        keyword_matches_df[
            ["file_name", "section_number",
             "section_label", "matched_heading",
             "text_excerpt"]
        ]
    )

Potential keyword-heading matches: 2


,file_name,section_number,section_label,matched_heading,text_excerpt
0,005_building-relationships-in-higher-education...,4,keywords,KEYWORDS,...
1,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,1,front_matter,Keywords,"urg, the Netherlands\n\n\nA R T I C L E I N F..."


## 16. Inspect potential author-keyword lists

The keyword-heading search identified two development documents with possible author-provided keywords.

The first document contains a keyword list embedded in a multi-column PDF layout. The second contains a Keywords heading within the front matter.

This step examines the surrounding original text before extracting or preserving the author keywords.

In [27]:
tetraselmis_match = keyword_matches_df[
    keyword_matches_df["file_name"].str.contains(
        "tetraselmis", case=False, na=False
    )
]

for _, row in tetraselmis_match.iterrows():
    document_id = row["document_id"]
    section_number = row["section_number"]

    section = dev_sections_df[
        (dev_sections_df["document_id"] == document_id) &
        (dev_sections_df["section_number"] == section_number)
    ].iloc[0]

    original_text = section["original_text"]

    import unicodedata
    normalized_text = unicodedata.normalize(
        "NFKC", original_text
    )

    heading_position = normalized_text.lower().find("keywords")

    print("Document:", row["file_name"])
    print("Section:", section_number)
    print("\n--- TEXT AROUND KEYWORDS HEADING ---\n")

    print(
        normalized_text[
            max(0, heading_position - 200):
            heading_position + 1200
        ]
    )

Document: 027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf
Section: 1

--- TEXT AROUND KEYWORDS HEADING ---

 Environment, Research Group Marine Biobased Chemistry, Joint Research Centre Zeeland,
Groene Woud 1a, 4331, NB, Middelburg, the Netherlands


A R T I C L E  I N F O                  A B S T R A C T

Keywords:                                  The growing demand of seafood alternatives is driven by concerns on overfishing, marine pollutants and animal
Microalgae                                        welfare in aquaculture and fisheries. Currently, the availability of non-animal-based seafood flavorings on the
Tetraselmis chuii                                market is limited, and animal-based seafood flavorings conflict with vegetarian and vegan criteria.
Seafood alternative                                           The aim of this study is to explore the use of Tetraselmis chuii as a seafood flavoring in a vegetable broth. TheSensory evaluati

### Author-keyword inspection findings

Two of the six development documents contain recognizable keyword headings in the extracted text.

- The higher-education publication contains a keyword list in a multi-column layout.
- The Tetraselmis publication contains a keyword heading in its front matter.
- Four other development documents have no keyword heading detected by the current search.

Keyword-heading detection is not sufficient to establish that author keywords are absent. Further validation against the original PDFs may be required.

Author keywords will be preserved separately from KeyBERT suggestions.

## 17. Record identified author keywords

The inspection identified author-keyword lists in two development publications.

The higher-education article contains five apparent author keywords in a multi-column PDF layout. The Tetraselmis article contains six apparent author keywords within the front matter.

These lists are recorded provisionally until verified against the original PDFs.

For documents without a confirmed keyword list, the author-keyword status remains unknown. They must not automatically be classified as having no author keywords.

In [28]:
author_keywords_by_file = {
    "005_building-relationships-in-higher-education-to-support-students-motivation.pdf": [
        "Student-staff interactions",
        "Need-supportive teaching",
        "Relationship quality",
        "Motivation",
        "Self-determination theory"
    ],
    "027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf": [
        "Microalgae",
        "Tetraselmis chuii",
        "Seafood alternative",
        "Sensory evaluation",
        "Consumer acceptance",
        "Umami"
    ]
}

# Create a separate record for every development document
author_records = []

for _, row in dev_df.iterrows():

    filename = row["file_name"]

    if filename in author_keywords_by_file:
        keywords = author_keywords_by_file[filename]
        status = "provisional"
    else:
        keywords = []
        status = "unknown"

    author_records.append({
        "document_id": row["document_id"],
        "file_name": filename,
        "language": row["language"],
        "author_keywords": keywords,
        "author_keyword_status": status,
        "author_keyword_count": len(keywords)
    })

author_keywords_df = pd.DataFrame(author_records)

display(
    author_keywords_df[
        ["file_name", "language",
         "author_keyword_count", "author_keyword_status"]
    ]
)

,file_name,language,author_keyword_count,author_keyword_status
0,005_building-relationships-in-higher-education...,en,5,provisional
1,072_overflow-tests-dutch-levees-annexes.pdf,en,0,unknown
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en,6,provisional
3,062_leven-in-zeeland-2021.pdf,nl,0,unknown
4,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl,0,unknown
5,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl,0,unknown


In [29]:
# Apply the business rule only when author-keyword
# presence or absence has been established.

def determine_suggestion_count(row):
    status = row["author_keyword_status"]

    if status in ["provisional", "verified"]:
        if row["author_keyword_count"] > 0:
            return 5

    if status == "confirmed_absent":
        return 10

    # Unknown status requires further inspection
    return None


author_keywords_df["required_suggestions"] = (
    author_keywords_df.apply(
        determine_suggestion_count,
        axis=1
    )
)

display(
    author_keywords_df[
        ["file_name", "author_keyword_status",
         "author_keyword_count", "required_suggestions"]
    ]
)

,file_name,author_keyword_status,author_keyword_count,required_suggestions
0,005_building-relationships-in-higher-education...,provisional,5,5.0
1,072_overflow-tests-dutch-levees-annexes.pdf,unknown,0,NaN
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,provisional,6,5.0
3,062_leven-in-zeeland-2021.pdf,unknown,0,NaN
4,063_living-lab-sporten-in-de-wijk-gemeente-mid...,unknown,0,NaN
5,049_eindrapportage-vitaliteitsakkoord-gemeente...,unknown,0,NaN


In [30]:
verified_files = [
    "005_building-relationships-in-higher-education-to-support-students-motivation.pdf",
    "027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf"
]

author_keywords_df.loc[
    author_keywords_df["file_name"].isin(verified_files),
    "author_keyword_status"
] = "verified"

display(
    author_keywords_df[
        ["file_name", "author_keyword_status",
         "required_suggestions"]
    ]
)

,file_name,author_keyword_status,required_suggestions
0,005_building-relationships-in-higher-education...,verified,5.0
1,072_overflow-tests-dutch-levees-annexes.pdf,unknown,NaN
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,verified,5.0
3,062_leven-in-zeeland-2021.pdf,unknown,NaN
4,063_living-lab-sporten-in-de-wijk-gemeente-mid...,unknown,NaN
5,049_eindrapportage-vitaliteitsakkoord-gemeente...,unknown,NaN


## 18. Keyword generation fallback

The project team agreed that when author-provided keywords cannot be reliably extracted, the system should proceed with keyword suggestions.

The processing rules are:

1. Preserve verified author keywords and generate five additional suggestions.
2. When author keywords are unavailable or extraction is unsuccessful, generate ten suggestions.
3. Keep author keywords and model suggestions in separate fields.
4. Record whether the document used the fallback procedure.
5. Do not interpret unsuccessful extraction as proof that a publication has no author keywords.

The fallback enables keyword generation for all development documents while retaining transparency about extraction limitations.

In [32]:
# Mark documents without verified author keywords
# as eligible for the agreed generation fallback.

fallback_mask = (
    author_keywords_df["author_keyword_status"] == "unknown"
)

author_keywords_df.loc[
    fallback_mask,
    "author_keyword_status"
] = "extraction_unavailable"

# Assign the number of new suggestions.
# Verified keywords -> 5 additional suggestions.
# Extraction unavailable -> 10 suggestions.

author_keywords_df["required_suggestions"] = (
    author_keywords_df["author_keyword_status"].map({
        "verified": 5,
        "extraction_unavailable": 10
    })
)

author_keywords_df["generation_mode"] = (
    author_keywords_df["author_keyword_status"].map({
        "verified": "additional_keywords",
        "extraction_unavailable": "fallback_generation"
    })
)

assert author_keywords_df["required_suggestions"].notna().all()

display(
    author_keywords_df[
        [
            "file_name",
            "language",
            "author_keyword_status",
            "required_suggestions",
            "generation_mode"
        ]
    ]
)

,file_name,language,author_keyword_status,required_suggestions,generation_mode
0,005_building-relationships-in-higher-education...,en,verified,5,additional_keywords
1,072_overflow-tests-dutch-levees-annexes.pdf,en,extraction_unavailable,10,fallback_generation
2,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,en,verified,5,additional_keywords
3,062_leven-in-zeeland-2021.pdf,nl,extraction_unavailable,10,fallback_generation
4,063_living-lab-sporten-in-de-wijk-gemeente-mid...,nl,extraction_unavailable,10,fallback_generation
5,049_eindrapportage-vitaliteitsakkoord-gemeente...,nl,extraction_unavailable,10,fallback_generation


## 19. Reuse multilingual stop-word resources

The existing preprocessing pipeline uses spaCy for English and Dutch stop-word handling.

To maintain consistency with the project, this KeyBERT experiment reuses spaCy's language-specific stop-word resources rather than introducing a separate NLTK-based implementation.

Stop words are used to restrict candidate keyword phrases. The original document text is retained for semantic embedding and similarity calculations.

Unicode normalization is applied separately to reduce PDF extraction artifacts.

In [35]:
import spacy

# Reuse the same language resources as Notebook 05.
nlp_en = spacy.blank("en")
nlp_nl = spacy.blank("nl")

# Use copies so we do not modify spaCy's defaults.
english_stopwords = set(nlp_en.Defaults.stop_words)
dutch_stopwords = set(nlp_nl.Defaults.stop_words)

# Preserve important negation words, consistent with Notebook 05.
PROTECTED_WORDS = {
    "en": {"no", "not", "nor", "never", "without"},
    "nl": {"niet", "geen", "nooit", "zonder", "noch"}
}

english_stopwords -= PROTECTED_WORDS["en"]
dutch_stopwords -= PROTECTED_WORDS["nl"]

print("English stop words:", len(english_stopwords))
print("Dutch stop words:", len(dutch_stopwords))
print("spaCy multilingual resources initialized.")

English stop words: 321
Dutch stop words: 339
spaCy multilingual resources initialized.


## 20. Generate multilingual KeyBERT keyword candidates

This step uses the existing multilingual KeyBERT model to extract keyword candidates from the six development documents.

English and Dutch stop-word lists are applied according to document language. Unicode NFKC normalization reduces PDF ligature artifacts.

The extraction uses one- and two-word phrases, Maximal Marginal Relevance (MMR) with diversity 0.3, and up to 30 ranked candidates per document.

The candidates will be reviewed and filtered before selecting five additional suggestions for documents with verified author keywords or ten suggestions for documents using the fallback procedure.

KeyBERT is an extractive method: its suggested phrases come from the source text rather than being newly generated by a language model.

In [36]:
import re
import unicodedata
from sklearn.feature_extraction.text import CountVectorizer

def normalize_keybert_text(text):
    """Normalize PDF text before keyword extraction."""
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_keybert_candidates(text, language):
    """Extract up to 30 multilingual keyword candidates."""

    cleaned_text = normalize_keybert_text(text)

    if not cleaned_text:
        return []

    if language == "nl":
        stopword_list = sorted(dutch_stopwords)
    else:
        stopword_list = sorted(english_stopwords)

    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=stopword_list,
        lowercase=True,
        token_pattern=r"(?u)\b\w\w+\b"
    )

    keywords = kw_model.extract_keywords(
        cleaned_text,
        vectorizer=vectorizer,
        use_mmr=True,
        diversity=0.3,
        top_n=30
    )

    return keywords


# First, test the function on ONE English document.
example = keybert_dev_df.iloc[0]

test_candidates = extract_keybert_candidates(
    example["source_text"],
    example["language"]
)

print("Document:", example["file_name"])
print("Language:", example["language"])
print("Candidates returned:", len(test_candidates))
print("\nTop 15 candidates:\n")

for rank, (keyword, score) in enumerate(
    test_candidates[:15], start=1
):
    print(f"{rank:2}. {keyword:40} {score:.4f}")

/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Document: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
Language: en
Candidates returned: 30

Top 15 candidates:

 1. students motivation                      0.6466
 2. students perceptions                     0.6320
 3. students motiv                           0.6269
 4. motivation students                      0.6216
 5. perceptions student                      0.6022
 6. motivation context                       0.5863
 7. associated motivation                    0.5846
 8. affect students                          0.5768
 9. motivation perceived                     0.5748
10. conclusion students                      0.5680
11. students intrin                          0.5601
12. motivation interplay                     0.5592
13. associations motivation                  0.5348
14. important students                       0.5327
15. relationship students                    0.5303


## 21. Filter and rank KeyBERT keyword suggestions

The initial KeyBERT results contain relevant concepts but also repeated phrases, truncated words, and PDF extraction artifacts.

This step applies conservative filtering to remove:

- Duplicate candidates.
- Candidates already present in verified author keywords.
- Phrases containing known section headings.
- Selected incomplete words observed in PDF extraction.

The remaining candidates retain their original KeyBERT relevance scores. Filtering improves candidate quality but does not guarantee semantic relevance or grammatical correctness.

The system will report when fewer than the requested number of suitable candidates remain.

In [37]:
import unicodedata
import re

# Known extraction artifacts identified during inspection.
# This is an initial, conservative list.
ARTIFACT_WORDS = {
    "motiv", "intrin", "insti",
    "conclusion", "introduction", "abstract",
    "discussion", "references",
    "samenvatting", "inleiding", "conclusie"
}

def normalize_keyword(keyword):
    """Normalize a keyword for duplicate comparison."""
    keyword = unicodedata.normalize("NFKC", keyword)
    keyword = keyword.casefold()
    keyword = re.sub(r"\s+", " ", keyword).strip()
    return keyword


def filter_keybert_candidates(
    candidates,
    author_keywords,
    required_count
):
    """Remove obvious artifacts and duplicate suggestions."""

    existing = {
        normalize_keyword(keyword)
        for keyword in author_keywords
    }

    selected = []
    seen = set(existing)

    for keyword, score in candidates:
        normalized = normalize_keyword(keyword)

        if not normalized:
            continue

        words = normalized.split()

        # Remove known PDF artifacts and section headings.
        if any(word in ARTIFACT_WORDS for word in words):
            continue

        # Remove exact duplicates of author keywords or
        # previously selected suggestions.
        if normalized in seen:
            continue

        selected.append({
            "keyword": keyword,
            "score": round(float(score), 4)
        })

        seen.add(normalized)

        if len(selected) >= required_count:
            break

    return selected


# Test filtering on the first development document.
example_author_row = author_keywords_df[
    author_keywords_df["document_id"] == example["document_id"]
].iloc[0]

filtered_example = filter_keybert_candidates(
    candidates=test_candidates,
    author_keywords=example_author_row["author_keywords"],
    required_count=5
)

print("Verified author keywords:")
for keyword in example_author_row["author_keywords"]:
    print(" -", keyword)

print("\nNew KeyBERT suggestions:")
for item in filtered_example:
    print(
        f" - {item['keyword']} "
        f"(score: {item['score']:.4f})"
    )

print("\nSuggestions returned:", len(filtered_example))

Verified author keywords:
 - Student-staff interactions
 - Need-supportive teaching
 - Relationship quality
 - Motivation
 - Self-determination theory

New KeyBERT suggestions:
 - students motivation (score: 0.6466)
 - students perceptions (score: 0.6320)
 - motivation students (score: 0.6216)
 - perceptions student (score: 0.6022)
 - motivation context (score: 0.5863)

Suggestions returned: 5


## 22. Reduce redundancy in keyword suggestions

The initial filtering removed exact duplicates and known PDF artifacts, but several suggestions remained semantically or lexically redundant.

This step introduces additional filtering to:

- Remove phrases that contain the same words in a different order.
- Exclude candidates that simply expand an existing author keyword with generic words.
- Reduce repeated concepts among the selected suggestions.
- Preserve the original KeyBERT relevance scores.

This is a heuristic approach. It does not guarantee that every selected keyword is meaningful or unique in a semantic sense.

In [38]:
# Generic words that often add little meaning to
# an existing author keyword.
GENERIC_WORDS = {
    "student", "students", "context", "associated",
    "important", "perceived", "results", "conclusion",
    "study", "research", "article", "paper",
    "onderzoek", "resultaten", "belangrijk"
}

def keyword_word_set(keyword):
    """Return normalized words for redundancy checks."""
    normalized = normalize_keyword(keyword)
    return set(normalized.split())


def filter_keybert_candidates_v2(
    candidates,
    author_keywords,
    required_count
):
    """Filter duplicates, artifacts, and obvious redundancy."""

    author_normalized = {
        normalize_keyword(keyword)
        for keyword in author_keywords
    }

    author_word_sets = [
        keyword_word_set(keyword)
        for keyword in author_keywords
    ]

    selected = []
    selected_word_sets = []
    seen = set()

    for keyword, score in candidates:
        normalized = normalize_keyword(keyword)
        words = normalized.split()
        word_set = set(words)

        if not words:
            continue

        if any(word in ARTIFACT_WORDS for word in words):
            continue

        if normalized in author_normalized:
            continue

        if normalized in seen:
            continue

        # Reject word-order duplicates.
        if word_set in selected_word_sets:
            continue

        # Reject simple expansions of author keywords
        # using only generic additional words.
        redundant_with_author = False

        for author_words in author_word_sets:
            if author_words and author_words.issubset(word_set):
                extra_words = word_set - author_words

                if not extra_words or extra_words.issubset(
                    GENERIC_WORDS
                ):
                    redundant_with_author = True
                    break

        if redundant_with_author:
            continue

        # Reject candidates that substantially overlap
        # with already selected keyword phrases.
        redundant_with_selected = False

        for previous_words in selected_word_sets:
            overlap = len(word_set & previous_words)
            smaller_size = min(
                len(word_set), len(previous_words)
            )

            if smaller_size > 0 and overlap / smaller_size >= 0.8:
                redundant_with_selected = True
                break

        if redundant_with_selected:
            continue

        selected.append({
            "keyword": keyword,
            "score": round(float(score), 4)
        })

        selected_word_sets.append(word_set)
        seen.add(normalized)

        if len(selected) >= required_count:
            break

    return selected


filtered_example_v2 = filter_keybert_candidates_v2(
    candidates=test_candidates,
    author_keywords=example_author_row["author_keywords"],
    required_count=5
)

print("Improved KeyBERT suggestions:\n")

for item in filtered_example_v2:
    print(
        f" - {item['keyword']} "
        f"(score: {item['score']:.4f})"
    )

print("\nSuggestions returned:", len(filtered_example_v2))

Improved KeyBERT suggestions:

 - students perceptions (score: 0.6320)
 - perceptions student (score: 0.6022)
 - affect students (score: 0.5768)
 - motivation interplay (score: 0.5592)
 - associations motivation (score: 0.5348)

Suggestions returned: 5


## 23. Inspect a larger KeyBERT candidate pool

The previous filtering experiments reduced exact duplication but still produced awkward or repetitive phrases.

This experiment extracts a larger pool of candidates and inspects their relevance and quality before finalizing the keyword selection procedure.

The purpose is to assess whether better suggestions are available without introducing excessive filtering rules.

In [39]:
# Inspect a larger candidate pool for the first document.

cleaned_text = normalize_keybert_text(example["source_text"])

stopword_list = (
    sorted(dutch_stopwords)
    if example["language"] == "nl"
    else sorted(english_stopwords)
)

inspection_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=stopword_list,
    lowercase=True,
    token_pattern=r"(?u)\b\w\w+\b"
)

inspection_candidates = kw_model.extract_keywords(
    cleaned_text,
    vectorizer=inspection_vectorizer,
    use_mmr=True,
    diversity=0.3,
    top_n=60
)

print("Candidates returned:", len(inspection_candidates))
print("\nTop 30 candidates:\n")

for rank, (keyword, score) in enumerate(
    inspection_candidates[:30], start=1
):
    print(f"{rank:2}. {keyword:40} {score:.4f}")

/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Candidates returned: 60

Top 30 candidates:

 1. students motivation                      0.6466
 2. students perceptions                     0.6320
 3. students motiv                           0.6269
 4. motivation students                      0.6216
 5. perceptions student                      0.6022
 6. motivation context                       0.5863
 7. associated motivation                    0.5846
 8. affect students                          0.5768
 9. motivation class                         0.5760
10. motivation associated                    0.5753
11. motivation perceived                     0.5748
12. motivation relationship                  0.5736
13. motivation association                   0.5722
14. motivation structure                     0.5721
15. conclusion students                      0.5680
16. students intrin                          0.5601
17. motivation interplay                     0.5592
18. identified motivation                    0.5566
19. motivation indi

## 24. Inspect source-text quality before further extraction

The larger KeyBERT candidate pool continued to contain truncated words, repeated concepts, and section-heading artifacts.

This step investigates whether these problems originate in the source text used for keyword extraction.

The original source text will be preserved. Any cleaning improvements will be applied to a separate working copy.

The goal is to improve input quality before repeating keyword extraction across the six development documents.

In [40]:
# Examine the source text around suspicious candidate words.

source_text = normalize_keybert_text(example["source_text"])

suspicious_terms = [
    "motiv",
    "intrin",
    "conclusion",
    "perceptions"
]

for term in suspicious_terms:
    print(f"\n{'=' * 60}")
    print(f"SEARCH TERM: {term}")
    print("=" * 60)

    matches = list(
        re.finditer(re.escape(term), source_text, flags=re.IGNORECASE)
    )

    print("Occurrences:", len(matches))

    for match in matches[:3]:
        start = max(0, match.start() - 90)
        end = min(len(source_text), match.end() + 110)

        print("\n...", source_text[start:end], "...")


SEARCH TERM: motiv
Occurrences: 28

... Conclusion Students’ motivation is ignited by their social context (e.g. Connell and Wellborn 1991; Ryan and Deci 2017; Skinner and Belm ...

... the HE-institute. Students’ perceptions of both social contexts are associated with their motivation. Especially the affective component of the student-staffrelationship quality was found to be important f ...

... omponent of the student-staffrelationship quality was found to be important for students’ motivation, as perceived affective commitment and involvement predicted students’ more autonomous forms of motivati ...

SEARCH TERM: intrin
Occurrences: 3

...  commitment and involvement predicted students’ more autonomous forms of motivation (i.e. intrin- sic and identified motivation). Structure was found to be negatively associated with con- trolled forms of m ...

... vement seemed to function as a mediator in the association between affective conflict and intrin- sic and identified motivation. Th

## 25. Clean PDF extraction artifacts

Inspection of the source text revealed words broken by PDF line wrapping, missing spaces around some compound expressions, and section headings merged into sentences.

This step creates a cleaned working copy of the source text.

The cleaning procedure:
- Normalizes Unicode characters.
- Rejoins words split by hyphenation, such as "intrin- sic".
- Corrects selected spacing artifacts identified during inspection.
- Removes a leading section heading when present.

The original source text remains unchanged. Cleaning is heuristic and may require further refinement for other documents.

In [41]:
def clean_pdf_text_for_keybert(text):
    """Clean observed PDF artifacts without changing source data."""

    if not isinstance(text, str):
        return ""

    # Normalize ligatures and other Unicode variants.
    text = unicodedata.normalize("NFKC", text)

    # Rejoin words broken by PDF line wrapping:
    # "intrin- sic" -> "intrinsic"
    # "stu- dents" -> "students"
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÿ])-\s+(?=[a-zà-ÿ])",
        "",
        text
    )

    # Fix a spacing artifact observed in this document.
    text = re.sub(
        r"\bstudent-staffrelationships\b",
        "student-staff relationships",
        text,
        flags=re.IGNORECASE
    )

    # Remove a section heading at the start of the text.
    text = re.sub(
        r"^\s*(?:Conclusion|Introduction|Abstract|Discussion)"
        r"(?=\s+[A-Z])\s+",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text


cleaned_example_text = clean_pdf_text_for_keybert(
    example["source_text"]
)

print("Original characters:", len(example["source_text"]))
print("Cleaned characters:", len(cleaned_example_text))

print("\nChecking previously observed artifacts:")
for phrase in [
    "intrin- sic",
    "stu- dents",
    "student-staffrelationships",
    "intrinsic",
    "students",
    "student-staff relationships"
]:
    print(f"{phrase:30} {phrase in cleaned_example_text}")

Original characters: 6000
Cleaned characters: 5997

Checking previously observed artifacts:
intrin- sic                    False
stu- dents                     False
student-staffrelationships     False
intrinsic                      True
students                       True
student-staff relationships    True


## 26. Compare KeyBERT results before and after PDF cleaning

This experiment repeats KeyBERT extraction using the cleaned source text from Section 25.

The model, English stop-word list, n-gram range, MMR diversity, and candidate count remain unchanged.

By keeping these settings constant, we can examine whether PDF text cleaning improves candidate quality.

The comparison is exploratory and uses one English development document. It does not yet establish overall performance across the multilingual dataset.

In [42]:
# Reuse the same KeyBERT settings as Section 23.
# Only the input text changes.

cleaned_candidates = kw_model.extract_keywords(
    cleaned_example_text,
    vectorizer=CountVectorizer(
        ngram_range=(1, 2),
        stop_words=sorted(english_stopwords),
        lowercase=True,
        token_pattern=r"(?u)\b\w\w+\b"
    ),
    use_mmr=True,
    diversity=0.3,
    top_n=60
)

print("Candidates returned:", len(cleaned_candidates))
print("\nTop 30 candidates AFTER cleaning:\n")

for rank, (keyword, score) in enumerate(
    cleaned_candidates[:30], start=1
):
    print(f"{rank:2}. {keyword:40} {score:.4f}")


print("\nPreviously problematic phrases:")
problematic_phrases = [
    "students motiv",
    "students intrin",
    "conclusion students"
]

for phrase in problematic_phrases:
    found = any(
        keyword == phrase
        for keyword, score in cleaned_candidates
    )
    print(f"{phrase:25} {'STILL PRESENT' if found else 'NOT PRESENT'}")

/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Candidates returned: 60

Top 30 candidates AFTER cleaning:

 1. students motivation                      0.6557
 2. motivation students                      0.6359
 3. students perceptions                     0.6283
 4. contexts students                        0.6143
 5. motivation inclass                       0.6127
 6. motivation context                       0.6000
 7. perceptions student                      0.5983
 8. associated motivation                    0.5949
 9. motivation relationship                  0.5918
10. motivation association                   0.5903
11. students intrinsic                       0.5888
12. affect students                          0.5878
13. motivation associated                    0.5855
14. motivation structure                     0.5830
15. motivation perceived                     0.5823
16. motivation interplay                     0.5815
17. motivation intrinsic                     0.5672
18. motivation embedded                      0.5635
19. 

## 27. Extract KeyBERT candidates for all development documents

This step applies the multilingual KeyBERT model to all six development documents (three English and three Dutch).

Each document uses:
- The previously prepared source text.
- Unicode normalization and PDF artifact cleaning.
- Language-specific spaCy stop words.
- One- and two-word candidate phrases.
- MMR with diversity 0.3.
- A pool of up to 60 candidates.

The results are stored separately from the verified author keywords. The six test documents remain excluded during development.

The candidate pool is not yet the final set of five or ten suggestions. Candidate quality and redundancy will be assessed before final selection.

In [43]:
# Extract candidates for all six development documents.

all_dev_candidates = []

for _, row in keybert_dev_df.iterrows():

    document_id = row["document_id"]
    language = row["language"]

    cleaned_text = clean_pdf_text_for_keybert(
        row["source_text"]
    )

    stopword_list = (
        sorted(dutch_stopwords)
        if language == "nl"
        else sorted(english_stopwords)
    )

    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=stopword_list,
        lowercase=True,
        token_pattern=r"(?u)\b\w\w+\b"
    )

    candidates = kw_model.extract_keywords(
        cleaned_text,
        vectorizer=vectorizer,
        use_mmr=True,
        diversity=0.3,
        top_n=60
    )

    for rank, (keyword, score) in enumerate(
        candidates, start=1
    ):
        all_dev_candidates.append({
            "document_id": document_id,
            "file_name": row["file_name"],
            "language": language,
            "rank": rank,
            "keyword": keyword,
            "score": round(float(score), 4)
        })

    print(
        f"Completed: {row['file_name'][:55]}... "
        f"({language}, {len(candidates)} candidates)"
    )


keybert_candidates_df = pd.DataFrame(
    all_dev_candidates
)

print("\nTotal candidates:", len(keybert_candidates_df))
print(
    "Documents processed:",
    keybert_candidates_df["document_id"].nunique()
)

print("\nCandidates by language:")
print(
    keybert_candidates_df.groupby("language")
    .size()
)

/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 005_building-relationships-in-higher-education-to-suppo... (en, 60 candidates)


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 072_overflow-tests-dutch-levees-annexes.pdf... (en, 60 candidates)


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-a... (en, 60 candidates)
Completed: 062_leven-in-zeeland-2021.pdf... (nl, 60 candidates)
Completed: 063_living-lab-sporten-in-de-wijk-gemeente-middelburg.p... (nl, 60 candidates)
Completed: 049_eindrapportage-vitaliteitsakkoord-gemeente-goes.pdf... (nl, 60 candidates)

Total candidates: 360
Documents processed: 6

Candidates by language:
language
en    180
nl    180
dtype: int64


## 28. Select KeyBERT suggestions using the business rules

This step selects keyword suggestions from the multilingual KeyBERT candidate pool.

For documents with verified author keywords, five additional suggestions are requested. For documents where author-keyword extraction was unavailable, ten suggestions are requested as a fallback.

Verified author keywords remain in a separate field and are never overwritten.

The existing redundancy filter is applied to the ranked candidates. If insufficient candidates remain, the shortage is reported rather than silently filling the list.

The output is a provisional development result requiring human relevance review.

In [44]:
# Apply the business rules to all six development documents.

all_dev_suggestions = []
selection_summary = []

for _, author_row in author_keywords_df.iterrows():

    document_id = author_row["document_id"]
    required_count = int(author_row["required_suggestions"])

    document_candidates = keybert_candidates_df[
        keybert_candidates_df["document_id"] == document_id
    ].sort_values("rank")

    candidate_pairs = list(
        zip(
            document_candidates["keyword"],
            document_candidates["score"]
        )
    )

    selected = filter_keybert_candidates_v2(
        candidates=candidate_pairs,
        author_keywords=author_row["author_keywords"],
        required_count=required_count
    )

    for rank, item in enumerate(selected, start=1):
        all_dev_suggestions.append({
            "document_id": document_id,
            "file_name": author_row["file_name"],
            "language": author_row["language"],
            "author_keyword_status": author_row["author_keyword_status"],
            "generation_mode": author_row["generation_mode"],
            "suggestion_rank": rank,
            "suggested_keyword": item["keyword"],
            "keybert_score": item["score"]
        })

    selection_summary.append({
        "file_name": author_row["file_name"],
        "language": author_row["language"],
        "author_keyword_status": author_row["author_keyword_status"],
        "required": required_count,
        "selected": len(selected),
        "shortfall": required_count - len(selected)
    })


keybert_suggestions_df = pd.DataFrame(
    all_dev_suggestions
)

selection_summary_df = pd.DataFrame(
    selection_summary
)

print("Selection summary:\n")
print(
    selection_summary_df[
        [
            "file_name",
            "language",
            "author_keyword_status",
            "required",
            "selected",
            "shortfall"
        ]
    ].to_string(index=False)
)

print("\nTotal suggestions selected:",
      len(keybert_suggestions_df))

print("Total suggestions required:",
      selection_summary_df["required"].sum())

print("Documents with shortages:",
      (selection_summary_df["shortfall"] > 0).sum())

Selection summary:

                                                                           file_name language  author_keyword_status  required  selected  shortfall
   005_building-relationships-in-higher-education-to-support-students-motivation.pdf       en               verified         5         5          0
                                         072_overflow-tests-dutch-levees-annexes.pdf       en extraction_unavailable        10        10          0
027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf       en               verified         5         5          0
                                                       062_leven-in-zeeland-2021.pdf       nl extraction_unavailable        10        10          0
                           063_living-lab-sporten-in-de-wijk-gemeente-middelburg.pdf       nl extraction_unavailable        10        10          0
                             049_eindrapportage-vitaliteitsakkoord-gemeente-goes.pdf       n

## 29. Review multilingual KeyBERT suggestions

This step displays the selected KeyBERT suggestions for all six development documents.

The review examines:
- Relevance to the document topic.
- Redundancy and awkward phrasing.
- PDF extraction artifacts.
- Language consistency.
- Overlap with verified author keywords.

The purpose is to assess the quality of the provisional suggestions before exporting the results or evaluating the reserved test documents.

All suggestions remain subject to human review.

In [45]:
# Display all selected suggestions by document.

for _, doc_row in author_keywords_df.iterrows():

    document_id = doc_row["document_id"]

    suggestions = keybert_suggestions_df[
        keybert_suggestions_df["document_id"] == document_id
    ].sort_values("suggestion_rank")

    print("\n" + "=" * 75)
    print("DOCUMENT:", doc_row["file_name"])
    print("LANGUAGE:", doc_row["language"])
    print("AUTHOR KEYWORD STATUS:", doc_row["author_keyword_status"])
    print("GENERATION MODE:", doc_row["generation_mode"])

    if doc_row["author_keyword_status"] == "verified":
        print("\nVerified author keywords:")
        for keyword in doc_row["author_keywords"]:
            print("  -", keyword)
    else:
        print("\nAuthor keywords: extraction unavailable")

    print("\nKeyBERT suggestions:")

    for _, suggestion in suggestions.iterrows():
        print(
            f"  {int(suggestion['suggestion_rank']):2}. "
            f"{suggestion['suggested_keyword']:<35} "
            f"{suggestion['keybert_score']:.4f}"
        )

    print(
        "\nSuggestions:",
        len(suggestions),
        "/",
        doc_row["required_suggestions"]
    )


DOCUMENT: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
LANGUAGE: en
AUTHOR KEYWORD STATUS: verified
GENERATION MODE: additional_keywords

Verified author keywords:
  - Student-staff interactions
  - Need-supportive teaching
  - Relationship quality
  - Motivation
  - Self-determination theory

KeyBERT suggestions:
   1. students perceptions                0.6283
   2. contexts students                   0.6143
   3. motivation inclass                  0.6127
   4. perceptions student                 0.5983
   5. motivation relationship             0.5918

Suggestions: 5 / 5

DOCUMENT: 072_overflow-tests-dutch-levees-annexes.pdf
LANGUAGE: en
AUTHOR KEYWORD STATUS: extraction_unavailable
GENERATION MODE: fallback_generation

Author keywords: extraction unavailable

KeyBERT suggestions:
   1. levees wl2022r20_047_2_p2c_overflow_tests_belgianlevee 0.6689
   2. levees wl2022r20_047_1_p2c_overflow_tests_belgianlevee 0.6680
   3. overflow tests           

## 30. Inspect document-section coverage

The first multilingual KeyBERT results satisfy the required suggestion counts, but manual inspection revealed redundancy, awkward phrasing, filename artifacts, and potentially unrepresentative keywords.

This step examines which document sections contribute to the 6,000-character KeyBERT input.

The objective is to determine whether section selection and truncation are limiting the topical relevance of the extracted keywords.

The existing source text and candidate results are preserved. No model parameters are changed in this step.

In [46]:
# Inspect section distribution for each development document.

for _, doc in keybert_dev_df.iterrows():

    doc_sections = selected_sections_df[
        selected_sections_df["document_id"] == doc["document_id"]
    ]

    print("\n" + "=" * 70)
    print("DOCUMENT:", doc["file_name"])
    print("LANGUAGE:", doc["language"])
    print("PREPARED TEXT LENGTH:", doc["source_characters"])

    print("\nAvailable selected sections:")
    print(
        doc_sections["section_label"]
        .value_counts()
        .to_string()
    )

    print("\nFirst 400 characters of KeyBERT input:")
    print(
        repr(doc["source_text"][:400])
    )


DOCUMENT: 005_building-relationships-in-higher-education-to-support-students-motivation.pdf
LANGUAGE: en
PREPARED TEXT LENGTH: 6000

Available selected sections:
section_label
discussion      6
results         5
methods         3
front_matter    3
conclusion      2

First 400 characters of KeyBERT input:
'Conclusion Students’ motivation is ignited by their social context (e.g. Connell and Wellborn 1991; Ryan and Deci 2017; Skinner and Belmont 1993). In HE, two embedded social contexts can be distinguished: in-class teaching approaches and student-staﬀrelationships in the HE-institute. Students’ perceptions of both social contexts are associated with their motivation. Especially the aﬀective compone'

DOCUMENT: 072_overflow-tests-dutch-levees-annexes.pdf
LANGUAGE: en
PREPARED TEXT LENGTH: 6000

Available selected sections:
section_label
introduction    100
unknown           6
front_matter      4

First 400 characters of KeyBERT input:
'1 Introduction Overflow tests on levees are design

## 31. Prepare balanced document text for KeyBERT

The initial document preparation prioritized sections and stopped after 6,000 characters. Manual review showed that this sometimes overrepresented introductions, forewords, methods, or table content.

This experiment creates an alternative input by distributing the available character budget across distinct section types.

The procedure:
- Uses the existing section labels and selected sections.
- Excludes references and tables of contents, as in the previous preparation.
- Takes text from multiple section types when available.
- Limits the total input to 6,000 characters.
- Applies the PDF-cleaning function from Section 25.

This is an exploratory KeyBERT-specific preparation method. It is not yet equivalent to the shared preprocessing used by the other keyword-extraction approaches.

In [50]:
# Prepare balanced document text for KeyBERT

MAX_BALANCED_CHARS = 6000

# 1. Find the actual text column in selected_sections_df.
#    The previous error happened because "text" was not a column.

possible_text_columns = [
    "original_text",
    "section_text",
    "text",
    "cleaned_text",
    "raw_text",
    "content",
    "section_content"
]

TEXT_COLUMN = next(
    (
        column
        for column in possible_text_columns
        if column in selected_sections_df.columns
    ),
    None
)

if TEXT_COLUMN is None:
    raise ValueError(
        "Could not identify the section-text column. "
        f"Available columns: {selected_sections_df.columns.tolist()}"
    )

print("Using section-text column:", TEXT_COLUMN)


# 2. Prepare balanced text for one document.

def prepare_balanced_keybert_text(document_id):
    """
    Distribute the available character budget across
    different section types.
    """

    doc_sections = selected_sections_df[
        selected_sections_df["document_id"] == document_id
    ].copy()

    if doc_sections.empty:
        return ""

    # Preserve the previously defined section priority.
    labels = [
        label
        for label in SECTION_PRIORITY
        if label in doc_sections["section_label"].values
    ]

    if not labels:
        return ""

    # Divide the character budget among available labels.
    chars_per_label = MAX_BALANCED_CHARS // len(labels)

    selected_texts = []

    for label in labels:

        label_sections = doc_sections[
            doc_sections["section_label"] == label
        ].sort_values("section_number")

        # Collect the text from this section type.
        label_text = " ".join(
            str(value)
            for value in label_sections[TEXT_COLUMN].fillna("")
        )

        # Apply the cleaning function from Section 25.
        label_text = clean_pdf_text_for_keybert(label_text)

        if label_text:
            selected_texts.append(
                label_text[:chars_per_label]
            )

    balanced_text = "\n\n".join(selected_texts)

    return balanced_text[:MAX_BALANCED_CHARS]


# 3. Create a separate balanced-text column.
#    The original source_text column remains unchanged.

keybert_dev_df["balanced_source_text"] = (
    keybert_dev_df["document_id"]
    .apply(prepare_balanced_keybert_text)
)

keybert_dev_df["balanced_source_characters"] = (
    keybert_dev_df["balanced_source_text"].str.len()
)


# 4. Display the results.

print("\nBalanced text preparation results:\n")

print(
    keybert_dev_df[
        [
            "file_name",
            "language",
            "source_characters",
            "balanced_source_characters"
        ]
    ].to_string(index=False)
)

print("\nDocuments processed:", len(keybert_dev_df))

print(
    "Documents with nonempty balanced text:",
    (keybert_dev_df["balanced_source_characters"] > 0).sum()
)

Using section-text column: original_text

Balanced text preparation results:

                                                                           file_name language  source_characters  balanced_source_characters
   005_building-relationships-in-higher-education-to-support-students-motivation.pdf       en               6000                        6000
                                         072_overflow-tests-dutch-levees-annexes.pdf       en               6000                        6000
027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf       en               6000                        6000
                                                       062_leven-in-zeeland-2021.pdf       nl               6000                        3023
                           063_living-lab-sporten-in-de-wijk-gemeente-middelburg.pdf       nl               6000                        5131
                             049_eindrapportage-vitaliteitsakkoord-gemeente-

## 32. Compare original and balanced KeyBERT inputs

This experiment compares keyword candidates extracted from the original priority-based text and the new balanced text.

Two development documents are selected because manual review identified clear quality problems:

- The English Tetraselmis broth article, where methods and table content dominated the initial suggestions.
- The Dutch Leven in Zeeland report, where introductory material produced overly generic suggestions.

The same multilingual model, language-specific stop words, n-gram range, MMR diversity, and candidate count are used.

This comparison is exploratory. It will help determine whether balancing section types improves keyword relevance and reduces artifacts.


In [51]:
# Compare original and balanced text

comparison_files = [
    "027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf",
    "062_leven-in-zeeland-2021.pdf"
]

for file_name in comparison_files:

    doc = keybert_dev_df[
        keybert_dev_df["file_name"] == file_name
    ].iloc[0]

    language = doc["language"]

    stopword_list = (
        sorted(dutch_stopwords)
        if language == "nl"
        else sorted(english_stopwords)
    )

    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=stopword_list,
        lowercase=True,
        token_pattern=r"(?u)\b\w\w+\b"
    )

    # Original candidates from Section 27
    original_candidates = (
        keybert_candidates_df[
            keybert_candidates_df["document_id"] == doc["document_id"]
        ]
        .sort_values("rank")
        .head(15)
    )

    # New candidates from balanced text
    balanced_candidates = kw_model.extract_keywords(
        doc["balanced_source_text"],
        vectorizer=vectorizer,
        use_mmr=True,
        diversity=0.3,
        top_n=60
    )

    print("\n" + "=" * 75)
    print("DOCUMENT:", file_name)
    print("LANGUAGE:", language)
    print("BALANCED TEXT LENGTH:", len(doc["balanced_source_text"]))

    print("\nTOP 15 — ORIGINAL INPUT")
    for _, row in original_candidates.iterrows():
        print(
            f"{int(row['rank']):2}. "
            f"{row['keyword']:<35} "
            f"{row['score']:.4f}"
        )

    print("\nTOP 15 — BALANCED INPUT")
    for rank, (keyword, score) in enumerate(
        balanced_candidates[:15], start=1
    ):
        print(
            f"{rank:2}. "
            f"{keyword:<35} "
            f"{score:.4f}"
        )


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(



DOCUMENT: 027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf
LANGUAGE: en
BALANCED TEXT LENGTH: 6000

TOP 15 — ORIGINAL INPUT
 1. ingredients broth                   0.7153
 2. broths powders                      0.6472
 3. broth powders                       0.6269
 4. broth fish                          0.6131
 5. broth contains                      0.5976
 6. fish broth                          0.5918
 7. broth powder                        0.5888
 8. chuii ingredients                   0.5831
 9. different broths                    0.5820
10. liquid broths                       0.5806
11. selected broths                     0.5746
12. fish tetraselmis                    0.5594
13. ingredients                         0.5522
14. cultivation tetraselmis             0.5512
15. flavor broth                        0.5457

TOP 15 — BALANCED INPUT
 1. ingredients broth                   0.7153
 2. broths containing                   0.6511
 3. broth powde

## 33. Final KeyBERT configuration and preprocessing decision

### Preprocessing decision

Two document-preparation strategies were compared:

1. Priority-based section selection with a maximum of 6,000 characters.
2. Balanced text allocation across available section labels.

The balanced approach produced some potentially useful additional phrases for the English Tetraselmis document, but did not clearly improve the Dutch Leven in Zeeland results.

Therefore, the priority-based preparation with PDF artifact cleaning is retained for the current KeyBERT implementation.

This decision is based on exploratory inspection of two development documents, not a statistically validated performance difference.

### Selected configuration

- Model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Keyword extraction: KeyBERT
- Languages: English and Dutch
- Candidate phrases: 1–2 words
- Stop words: language-specific spaCy lists
- Ranking: Maximal Marginal Relevance (MMR)
- MMR diversity: 0.3
- Candidate pool: up to 60 per document
- Maximum input length: 6,000 characters
- PDF cleaning: Unicode normalization and selected artifact corrections
- Selection: heuristic redundancy and artifact filtering

### Business rules

Verified author keywords are preserved separately.

Documents with verified author keywords receive five additional KeyBERT suggestions. Documents where author-keyword extraction is unavailable receive ten fallback suggestions and retain the `extraction_unavailable` flag.

### Limitations

Keyword quality remains inconsistent. Some candidates contain awkward word order, repeated concepts, or PDF extraction artifacts.

The author-keyword extraction process is not yet fully automated. Verified author keywords were manually identified for two development documents.

The final suggestions require human review before being used as production metadata.

Formal comparisons with YAKE or generative models require identical input preparation and evaluation procedures.

In [52]:
# Record selected KeyBERT configuration

keybert_config = {
    "model": MODEL_NAME,
    "method": "KeyBERT",
    "languages": ["en", "nl"],
    "preprocessing": "priority_based_with_pdf_cleaning",
    "max_source_characters": 6000,
    "ngram_range": [1, 2],
    "stopword_source": "spaCy",
    "ranking": "MMR",
    "diversity": 0.3,
    "candidate_pool_size": 60,
    "verified_author_keyword_suggestions": 5,
    "extraction_unavailable_fallback_suggestions": 10,
    "human_review_required": True
}

print("Final configuration recorded:\n")

for setting, value in keybert_config.items():
    print(f"{setting}: {value}")

Final configuration recorded:

model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
method: KeyBERT
languages: ['en', 'nl']
preprocessing: priority_based_with_pdf_cleaning
max_source_characters: 6000
ngram_range: [1, 2]
stopword_source: spaCy
ranking: MMR
diversity: 0.3
candidate_pool_size: 60
verified_author_keyword_suggestions: 5
extraction_unavailable_fallback_suggestions: 10
human_review_required: True


## 34. Export development results

This step exports the multilingual KeyBERT development results to reproducible files.

The exports include the complete candidate pool, provisional keyword suggestions, author-keyword information, selection counts, and the selected model configuration.

Verified author keywords are preserved separately from suggested keywords. Documents with unavailable author-keyword extraction retain the `extraction_unavailable` status.

These outputs are development results, not final validated metadata. Keyword relevance and redundancy still require human evaluation.

The reserved test documents are excluded from this export.

In [53]:
# Export development results

import json
from pathlib import Path

# Create output folder.
output_dir = PROJECT_ROOT / "output" / "keybert"
output_dir.mkdir(parents=True, exist_ok=True)

# Safety check: export development documents only.
dev_ids = set(dev_df["document_id"])
test_ids = set(test_df["document_id"])

assert dev_ids.isdisjoint(test_ids)

assert set(keybert_candidates_df["document_id"]) == dev_ids
assert set(keybert_suggestions_df["document_id"]) == dev_ids
assert set(author_keywords_df["document_id"]) == dev_ids

# 1. Save all keyword candidates.
keybert_candidates_df.to_csv(
    output_dir / "dev_candidates.csv",
    index=False,
    encoding="utf-8"
)

# 2. Save selected keyword suggestions.
keybert_suggestions_df.to_csv(
    output_dir / "dev_suggestions.csv",
    index=False,
    encoding="utf-8"
)

# 3. Save the selection summary.
selection_summary_df.to_csv(
    output_dir / "dev_selection_summary.csv",
    index=False,
    encoding="utf-8"
)

# 4. Save author keywords as JSON.
# JSON preserves each document's keyword list.

author_keyword_records = author_keywords_df.to_dict(
    orient="records"
)

with open(
    output_dir / "dev_author_keywords.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        author_keyword_records,
        file,
        ensure_ascii=False,
        indent=2
    )

# 5. Save final model configuration.
with open(
    output_dir / "keybert_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        keybert_config,
        file,
        ensure_ascii=False,
        indent=2
    )

# 6. Confirm saved files.
print("Development results exported successfully!\n")

for file_path in sorted(output_dir.iterdir()):
    if file_path.is_file():
        print(
            f"{file_path.name:35} "
            f"{file_path.stat().st_size:,} bytes"
        )

print("\nOutput folder:", output_dir)
print("Development documents:", len(dev_ids))
print("Test documents included: 0")

Development results exported successfully!

dev_author_keywords.json            2,342 bytes
dev_candidates.csv                  40,500 bytes
dev_selection_summary.csv           596 bytes
dev_suggestions.csv                 7,606 bytes
keybert_config.json                 494 bytes

Output folder: /Users/fatimahabbasi/Downloads/DDD/output/keybert
Development documents: 6
Test documents included: 0


## 35. Evaluate development keyword quality

This step evaluates the provisional KeyBERT suggestions from the six development documents.

The evaluation checks:
- Whether each document received the required number of suggestions.
- Whether selected suggestions contain exact duplicates.
- Whether selected suggestions contain the same words in different orders.
- Whether any suggestions exactly duplicate verified author keywords.
- Whether selected suggestions contain known PDF artifacts.

These checks are automated quality indicators, not a complete evaluation of relevance.

Manual inspection identified additional problems, including awkward word order, semantically repetitive phrases, and document-specific extraction artifacts. These limitations must be considered when interpreting the results.

Precision@5, Precision@10, and semantic relevance cannot be reliably calculated without an appropriate reference set or human relevance judgments.

In [54]:
# Automated development quality checks

quality_results = []

for _, doc in author_keywords_df.iterrows():

    document_id = doc["document_id"]

    suggestions = keybert_suggestions_df[
        keybert_suggestions_df["document_id"] == document_id
    ]["suggested_keyword"].tolist()

    normalized = [
        normalize_keyword(keyword)
        for keyword in suggestions
    ]

    word_sets = [
        frozenset(keyword_word_set(keyword))
        for keyword in suggestions
    ]

    author_normalized = {
        normalize_keyword(keyword)
        for keyword in doc["author_keywords"]
    }

    artifact_count = sum(
        any(
            word in ARTIFACT_WORDS
            for word in keyword.split()
        )
        for keyword in normalized
    )

    quality_results.append({
        "file_name": doc["file_name"],
        "language": doc["language"],
        "required": int(doc["required_suggestions"]),
        "selected": len(suggestions),
        "exact_duplicates": len(normalized) - len(set(normalized)),
        "word_order_duplicates": len(word_sets) - len(set(word_sets)),
        "author_exact_overlaps": sum(
            keyword in author_normalized
            for keyword in normalized
        ),
        "known_artifacts": artifact_count
    })


quality_df = pd.DataFrame(quality_results)

print("Development quality checks:\n")

print(
    quality_df.to_string(index=False)
)

print("\nOverall totals:")
print("Suggestions:", quality_df["selected"].sum())
print("Exact duplicates:", quality_df["exact_duplicates"].sum())
print(
    "Word-order duplicates:",
    quality_df["word_order_duplicates"].sum()
)
print(
    "Exact author-keyword overlaps:",
    quality_df["author_exact_overlaps"].sum()
)
print(
    "Known artifact matches:",
    quality_df["known_artifacts"].sum()
)

quality_df.to_csv(
    output_dir / "dev_quality_checks.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: dev_quality_checks.csv")

Development quality checks:

                                                                           file_name language  required  selected  exact_duplicates  word_order_duplicates  author_exact_overlaps  known_artifacts
   005_building-relationships-in-higher-education-to-support-students-motivation.pdf       en         5         5                 0                      0                      0                0
                                         072_overflow-tests-dutch-levees-annexes.pdf       en        10        10                 0                      0                      0                0
027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf       en         5         5                 0                      0                      0                0
                                                       062_leven-in-zeeland-2021.pdf       nl        10        10                 0                      0                      0              

## 36. Run the fixed KeyBERT pipeline on the reserved test documents

The final KeyBERT configuration is applied to the six previously reserved test documents: three English and three Dutch.

The test documents were excluded from development and parameter selection.

The pipeline uses the fixed priority-based text preparation, PDF cleaning, language-specific spaCy stop words, one- and two-word candidates, MMR diversity 0.3, and up to 60 candidates per document.

Because author keywords have not been verified for these test documents, the fallback procedure requests ten provisional suggestions per document and records `extraction_unavailable`. This status does not establish that author keywords are absent.

The test results will be exported without modifying the configuration based on their quality.

In [57]:
# Run the fixed KeyBERT pipeline on test documents

# 1. Prepare the six reserved test documents.
# The original preparation function uses selected_sections_df,
# which currently contains only development documents.
# We must temporarily provide the test sections instead.

# Select test sections from the full preprocessing dataset.
test_selected_sections_df = sections_with_language_df[
    sections_with_language_df["document_id"].isin(
        test_df["document_id"]
    )
    &
    sections_with_language_df["section_label"].isin(
        SECTION_PRIORITY
    )
].copy()

# Apply the same section priorities used during development.
priority_mapping = {
    label: index
    for index, label in enumerate(SECTION_PRIORITY)
}

test_selected_sections_df["priority"] = (
    test_selected_sections_df["section_label"]
    .map(priority_mapping)
)

test_selected_sections_df = (
    test_selected_sections_df.sort_values(
        ["document_id", "priority", "section_number"]
    )
)

# Save the original development section selection.
dev_selected_sections_df = selected_sections_df.copy()

# Temporarily use test sections with the existing function.
selected_sections_df = test_selected_sections_df

try:
    keybert_test_df = test_df[
        ["document_id", "file_name", "language"]
    ].copy()

    keybert_test_df["source_text"] = (
        keybert_test_df["document_id"]
        .apply(prepare_keybert_text)
    )

    keybert_test_df["source_characters"] = (
        keybert_test_df["source_text"].str.len()
    )

finally:
    # Restore development sections, even if an error occurs.
    selected_sections_df = dev_selected_sections_df


# 2. Check that all six test inputs are ready.

print("TEST INPUT PREPARATION:\n")

print(
    keybert_test_df[
        ["file_name", "language", "source_characters"]
    ].to_string(index=False)
)

dev_ids = set(dev_df["document_id"])
test_ids = set(test_df["document_id"])

assert len(keybert_test_df) == 6, (
    "Expected exactly six test documents."
)

assert dev_ids.isdisjoint(test_ids), (
    "Development and test documents overlap."
)

assert (
    keybert_test_df["source_characters"] > 0
).all(), (
    "At least one test document still has empty source text."
)

print("\nAll six test documents have nonempty input.")


# 3. Extract candidates and select suggestions.

test_candidate_records = []
test_suggestion_records = []
test_summary_records = []

for _, row in keybert_test_df.iterrows():

    document_id = row["document_id"]
    language = row["language"]

    # Apply the same PDF cleaning used in development.
    cleaned_text = clean_pdf_text_for_keybert(
        row["source_text"]
    )

    # Choose the correct language-specific stop words.
    stopword_list = (
        sorted(dutch_stopwords)
        if language == "nl"
        else sorted(english_stopwords)
    )

    # Same candidate settings as the development experiment.
    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=stopword_list,
        lowercase=True,
        token_pattern=r"(?u)\b\w\w+\b"
    )

    # Extract up to 60 candidates.
    candidates = kw_model.extract_keywords(
        cleaned_text,
        vectorizer=vectorizer,
        use_mmr=True,
        diversity=0.3,
        top_n=60
    )

    # Store all extracted candidates.
    for rank, (keyword, score) in enumerate(
        candidates, start=1
    ):
        test_candidate_records.append({
            "document_id": document_id,
            "file_name": row["file_name"],
            "language": language,
            "rank": rank,
            "keyword": keyword,
            "score": round(float(score), 4)
        })

    # Author keywords have not been verified for these
    # test documents, so use the agreed fallback.
    # This does NOT mean the documents have no author keywords.
    required_count = 10

    selected = filter_keybert_candidates_v2(
        candidates=candidates,
        author_keywords=[],
        required_count=required_count
    )

    # Store selected suggestions.
    for rank, item in enumerate(selected, start=1):

        test_suggestion_records.append({
            "document_id": document_id,
            "file_name": row["file_name"],
            "language": language,
            "author_keyword_status": "extraction_unavailable",
            "generation_mode": "fallback_generation",
            "suggestion_rank": rank,
            "suggested_keyword": item["keyword"],
            "keybert_score": item["score"]
        })

    # Record how many suggestions were selected.
    test_summary_records.append({
        "document_id": document_id,
        "file_name": row["file_name"],
        "language": language,
        "author_keyword_status": "extraction_unavailable",
        "required": required_count,
        "selected": len(selected),
        "shortfall": required_count - len(selected)
    })

    print(
        f"Completed: {row['file_name'][:55]}... "
        f"({language}, {len(candidates)} candidates, "
        f"{len(selected)}/10 suggestions)"
    )


# 4. Convert results into DataFrames.

keybert_test_candidates_df = pd.DataFrame(
    test_candidate_records
)

keybert_test_suggestions_df = pd.DataFrame(
    test_suggestion_records
)

test_summary_df = pd.DataFrame(
    test_summary_records
)


# 5. Check that test results contain no development documents.

assert set(
    keybert_test_candidates_df["document_id"]
).issubset(test_ids)

assert set(
    keybert_test_suggestions_df["document_id"]
).issubset(test_ids)

assert (
    test_summary_df["shortfall"] >= 0
).all()


# 6. Export test results to separate files.

output_dir = PROJECT_ROOT / "output" / "keybert"
output_dir.mkdir(parents=True, exist_ok=True)

keybert_test_candidates_df.to_csv(
    output_dir / "test_candidates.csv",
    index=False,
    encoding="utf-8"
)

keybert_test_suggestions_df.to_csv(
    output_dir / "test_suggestions.csv",
    index=False,
    encoding="utf-8"
)

test_summary_df.to_csv(
    output_dir / "test_selection_summary.csv",
    index=False,
    encoding="utf-8"
)


# 7. Display the final test results.

print("\n" + "=" * 75)
print("FINAL TEST RESULTS SUMMARY")
print("=" * 75)

print(
    test_summary_df[
        [
            "file_name",
            "language",
            "required",
            "selected",
            "shortfall"
        ]
    ].to_string(index=False)
)

print("\nTest documents:", len(test_summary_df))
print("Total candidates:", len(keybert_test_candidates_df))
print("Total suggestions:", len(keybert_test_suggestions_df))
print("Required suggestions:", test_summary_df["required"].sum())

print(
    "Documents with shortages:",
    (test_summary_df["shortfall"] > 0).sum()
)

print("\nFiles saved:")

for filename in [
    "test_candidates.csv",
    "test_suggestions.csv",
    "test_selection_summary.csv"
]:
    print(" -", output_dir / filename)

print("\nSection 36 completed successfully!")

TEST INPUT PREPARATION:

                                                                                 file_name language  source_characters
  017_integrating-text-structure-instruction-in-science-education-a-design-based-study.pdf       en               6000
                                             071_overflow-tests-belgian-levees-annexes.pdf       en               6000
020_long-term-forecast-of-thermal-mortality-with-climate-warming-in-riverine-amphipods.pdf       en               6000
                                                         056_jonge-zeeuwen-in-beweging.pdf       nl               6000
                          092_zeeuwse-inwoners-en-het-verduurzamen-van-de-eigen-woning.pdf       nl               6000
                                       047_efmzv-rendementsverbetering-oesterproductie.pdf       nl               4504

All six test documents have nonempty input.


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 017_integrating-text-structure-instruction-in-science-e... (en, 60 candidates, 10/10 suggestions)


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 071_overflow-tests-belgian-levees-annexes.pdf... (en, 60 candidates, 10/10 suggestions)


/opt/anaconda3/envs/kmp-keybert/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Completed: 020_long-term-forecast-of-thermal-mortality-with-climat... (en, 60 candidates, 10/10 suggestions)
Completed: 056_jonge-zeeuwen-in-beweging.pdf... (nl, 60 candidates, 10/10 suggestions)
Completed: 092_zeeuwse-inwoners-en-het-verduurzamen-van-de-eigen-w... (nl, 60 candidates, 10/10 suggestions)
Completed: 047_efmzv-rendementsverbetering-oesterproductie.pdf... (nl, 60 candidates, 10/10 suggestions)

FINAL TEST RESULTS SUMMARY
                                                                                 file_name language  required  selected  shortfall
  017_integrating-text-structure-instruction-in-science-education-a-design-based-study.pdf       en        10        10          0
                                             071_overflow-tests-belgian-levees-annexes.pdf       en        10        10          0
020_long-term-forecast-of-thermal-mortality-with-climate-warming-in-riverine-amphipods.pdf       en        10        10          0
                                      

## 37. Final implementation summary

### Objective

Implement and evaluate multilingual keyword extraction using KeyBERT for English and Dutch research PDFs in the Knowledge Management Platform project.

### Implementation

The pipeline uses the multilingual sentence-transformer model `paraphrase-multilingual-MiniLM-L12-v2` with KeyBERT.

Document text is selected using predefined section priorities, limited to 6,000 characters, and cleaned to reduce selected PDF extraction artifacts.

Candidate keywords contain one or two words. Language-specific stop words are applied, and Maximal Marginal Relevance (MMR) with diversity 0.3 is used to rank up to 60 candidates per document.

A heuristic filtering stage removes some duplicate and artifact-prone candidates.

### Author-keyword handling

Existing author keywords are stored separately when verified.

- Verified author keywords: generate five additional suggestions.
- Author-keyword extraction unavailable: generate ten fallback suggestions and retain the `extraction_unavailable` status.

The fallback status does not mean that the document has no author keywords.

### Development results

Six development documents were processed: three English and three Dutch.

- Candidates extracted: 360
- Provisional suggestions: 50
- Documents with suggestion shortages: 0
- Exact duplicates among selected suggestions: 0
- Word-order duplicates among selected suggestions: 0
- Exact overlaps with verified author keywords: 0
- Matches against the limited known-artifact word list: 0

### Reserved test results

Six reserved test documents were processed: three English and three Dutch.

- Candidates extracted: 360
- Provisional suggestions: 60
- Documents with suggestion shortages: 0

The model configuration was fixed before running the reserved test set.

### Limitations

The automated checks do not establish semantic relevance or metadata quality.

Manual inspection of development outputs identified awkward phrases, redundant concepts, and PDF-derived artifacts that the automated checks did not detect.

Author-keyword extraction was manually verified for two development documents and remains unavailable for the other development documents and all six test documents.

The implementation is an exploratory KeyBERT baseline, not a production-ready keyword enrichment system.

Formal comparison with YAKE and generative models requires consistent document inputs, shared evaluation criteria, and human relevance judgments.

### Output files

Results are saved in `output/keybert/`, including development and test candidates, selected suggestions, selection summaries, development quality checks, verified author-keyword records, and the model configuration.

### Conclusion

The multilingual KeyBERT pipeline successfully generated the required number of provisional suggestions for all 12 selected development and test documents.

Further work should prioritize author-keyword extraction, PDF text quality, human evaluation of relevance, and fair comparison with alternative methods.